In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc

from sklearn.model_selection import train_test_split

from cellina import CellinaModel
from utils import set_seed

# Get dataset

In [ ]:
set_seed(0)

In [ ]:
import os
DATA_ROOT = os.environ.get("DATA_ROOT", ".")
adata = sc.read(
    os.path.join(DATA_ROOT, f"datasets/cosmx/crc_wt_cosmx/crc_210.h5ad"),
    backup_url=f"https://zenodo.org/records/15574384/files/210.h5ad?download=1"
)
# adata.obsm = {} # NOTE: only some strange PCA embeddings are stored in obsm, which we don't need
adata.obs_names_make_unique()

label_to_coarse = {
    "epi1": "Epithelial",
    "epi2": "Epithelial",
    "epi3": "Epithelial",
    "epi4": "Epithelial",
    
    "fib1": "Fibroblast",
    "fib2": "Fibroblast",
    
    "EC": "Endothelial",
    "SMC": "Smooth_muscle",
    
    "BC": "B_cell",
    "PC_IgA": "Plasma_cell",
    "PC_IgG": "Plasma_cell",
    "PC_IgM": "Plasma_cell",
    
    "TC": "T_cell",
    
    "mye1": "Myeloid",
    "mye2": "Myeloid",
    
    "mast": "Mast_cell",
}

adata.obs["coarse_type"] = adata.obs['ist'].map(label_to_coarse)
labels_key = 'coarse_type'
domains_key = 'typ'
batch_key = 'sid'
adata = adata[~adata.obs[domains_key].isna()] # NOTE: Interesting to annotate?
adata = adata[~adata.obs[labels_key].isna()]

sc.pp.filter_cells(adata, min_counts=3)
sc.pp.filter_genes(adata, min_counts=3)

In [ ]:
adata.obs[labels_key] = adata.obs[labels_key].astype('category')
adata.obsm['spatial'] = adata.obs[['CenterX_global_px', 'CenterY_global_px']].values
adata.layers['counts'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, layer='counts', flavor='seurat_v3', n_top_genes=2000, subset=True)

In [ ]:
from cellina._spatial_utils import spatial_neighbors

n_neighbors = 50
spatial_neighbors(adata, bandwidth=np.inf, max_neighbours=n_neighbors, standardize=False)

In [ ]:
import pandas as pd
import numpy as np

import requests
import io

# read csv from link
# https://github.com/ventolab/cellphonedb-data/blob/master/data/interaction_input.csv
resource = requests.get('https://raw.githubusercontent.com/ventolab/cellphonedb-data/master/data/interaction_input.csv').content
resource = io.StringIO(resource.decode('utf-8'))
resource = pd.read_csv(resource, sep=',')
# keep only PPIs
resource = resource[resource['is_ppi']][['interactors']]
# replace + with _
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('+', '_'))
# if interactors contains two '-' replace the first one with '&
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('-', '&', 1) if x.count('-') == 2 else x)
# split by - and expand
resource = resource['interactors'].str.split('-', expand=True)
# replace & with - in the first column
resource[0] = resource[0].apply(lambda x: x.replace('&', '-'))
resource.columns = ['ligand', 'receptor']


In [ ]:
# Split ligands on '_' to get list of subunits
resource['ligand'] = resource['ligand'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('ligand').reset_index(drop=True)
ligands = resource['ligand'].unique()

In [ ]:
# Split receptors on '_' to get list of subunits
resource['receptor'] = resource['receptor'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('receptor').reset_index(drop=True)
receptors = resource['receptor'].unique()

In [ ]:
ligands_in_data = [g for g in ligands if g in adata.var_names]
receptors_in_data = [g for g in receptors if g in adata.var_names]

In [ ]:
use_ligands = True
use_receptors = False

neighbor_features = adata.var_names
if use_ligands:
    neighbor_features = ligands_in_data
if use_receptors:
    neighbor_features = neighbor_features + receptors_in_data

In [ ]:
adata.obs['neighbor'] = adata.obsp['spatial_connectivities'][:,0].toarray().astype(np.float32)

In [ ]:
from scipy.sparse import csr_matrix

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obsm['spatial_x'] = adata.obsp['spatial_connectivities'] @ adata[:, neighbor_features].X / n_neighbors
# float32
adata.obsm['spatial_x'] = csr_matrix(adata.obsm['spatial_x']).astype(np.float32)

In [ ]:
adata.uns['spatial_x_names'] = neighbor_features

In [ ]:
sc.pl.spatial(adata, color=[domains_key, labels_key], spot_size=100)

## Data splits

In [ ]:
split = "random"

# Get holdout indices
if split == "random":
    fraction = 0.1
    n_cells = adata.n_obs
    n_holdout = int(n_cells * fraction)

    # Randomly choose cells
    test_idx = np.random.choice(n_cells, n_holdout, replace=False)

elif split == "ood":
    # OOD: Non-ref, non-epithelial
    #is_tumor_region  = adata.obs["typ"].str.contains("CRC|TVA", regex=True)
    is_tumor_region  = adata.obs["typ"].str.contains("CRC", regex=True)
    is_non_epi = adata.obs["coarse_type"] != "Epithelial"

    # Combine for test set
    test_mask = (is_tumor_region) & (is_non_epi)
    test_idx = np.where(test_mask)[0]
else:
    raise ValueError(f"Unknown split: {split}")

# Get train/val indices
all_idx = np.arange(adata.n_obs)
trainval_idx = np.setdiff1d(all_idx, test_idx)

In [ ]:
# Set 'is_holdout' to False by default, then True for selected cells
adata.obs['is_holdout'] = False
adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

In [ ]:
validation_size = 0.1
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=validation_size,
    random_state=0,
    shuffle=True,
)

# Train

In [ ]:
base_path = os.path.join(DATA_ROOT, "data/cellina-reproducibility")

In [ ]:
model_args = {
    'adata': adata,
    'n_latent': 64,
    'use_observed_lib_size': True,
    'condition_on_intrinsic': False
    }
train_args = {'max_epochs': 100,
              'batch_size': 4096,
              'check_val_every_n_epoch': 1,
              'early_stopping': True,
              'early_stopping_patience': 25,
              'early_stopping_monitor': "validation_loss",
              'enable_checkpointing': True,
              'devices': [1],
              'datasplitter_kwargs': {
                  "external_indexing": [train_idx, val_idx, test_idx],
                  },
              }
plan_kwargs = {
    'lr': 1e-4,
    'normalize_losses': True
    }

In [ ]:
CellinaModel.setup_anndata(adata,
                           batch_key=batch_key,
                           labels_key=labels_key, 
                           domains_key=domains_key, 
                           spatial_obsm_key="spatial_x",
                           layer='counts')

In [ ]:
model = CellinaModel(
    **model_args,
    classifier_lambda=1., discriminator_lambda=1., 
)
model.train(
    **train_args,
    plan_kwargs=plan_kwargs
)

model.save(f"{base_path}/trained/crc_210_{split}", overwrite=True)

# Analysis

In [ ]:
save_path = f"{base_path}/trained/crc_210_{split}"
model = CellinaModel.load(save_path, adata)

## Latent visualization

In [ ]:
adata.obsm['cellina_basal'] = model.get_latent_representation(latent_key='z')
adata.obsm['cellina_spatial'] = model.get_latent_representation(latent_key='s')
adata.obsm['cellina_joint'] = model.get_latent_representation()

In [ ]:
sc.pp.neighbors(adata, use_rep='cellina_basal')
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=[domains_key, labels_key, 'ist', 'is_holdout'], wspace=0.4)

In [ ]:
sc.pp.neighbors(adata, use_rep='cellina_spatial')
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=[domains_key, labels_key, 'ist', 'is_holdout'], wspace=0.4)

In [ ]:
sc.pp.neighbors(adata, use_rep='cellina_joint')
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=[domains_key, labels_key, 'ist', 'is_holdout'], wspace=0.4)

## Latents cell type and niche prediction

In [ ]:
from utils import train_linear
from plotting import plot_roc_curves, plot_confusion_matrix

In [ ]:
test_size = 0.1
random_state = 0
latent_key = 'cellina_spatial'

In [ ]:
ct_mask = adata.obs['coarse_type'].isin(['Epithelial'])

In [ ]:
adata_sub = adata[(ct_mask) & ~adata.obs['jst'].isna()].copy()

In [ ]:
X = adata_sub.obsm[latent_key]

### Cell types

In [ ]:
y = adata_sub.obs['jst'].astype("category", copy=True).cat.codes
label_mapping = dict(enumerate(adata_sub.obs['jst'].astype("category").cat.categories))

In [ ]:
# stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=random_state
)

In [ ]:
clf_ct = train_linear(X=X_train, y=y_train, cv_folds=2, n_perm=0)

In [ ]:
# Get predicted probabilities from trained model
y_probs = clf_ct.predict_proba(X_test)
plot_roc_curves(y_true=y_test, y_probs=y_probs, label_mapping=label_mapping, macro_avg=True)

In [ ]:
y_pred = clf_ct.predict(X_test)
plot_confusion_matrix(y_true=y_test, y_pred=y_pred, label_mapping=label_mapping, normalize=True)

### Niches

In [ ]:
y = adata_sub.obs['typ'].astype("category").cat.codes
label_mapping = dict(enumerate(adata_sub.obs['typ'].astype("category").cat.categories))

In [ ]:
# stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=random_state
)

In [ ]:
clf_niche = train_linear(X=X_train, y=y_train, cv_folds=2, n_perm=0)

In [ ]:
y_probs = clf_niche.predict_proba(X_test)
plot_roc_curves(y_true=y_test, y_probs=y_probs, label_mapping=label_mapping, macro_avg=True)

In [ ]:
y_pred = clf_niche.predict(X_test)
plot_confusion_matrix(y_true=y_test, y_pred=y_pred, label_mapping=label_mapping, normalize=True)

## Hotspot

In [ ]:
import pandas as pd
import decoupler as dc
import hotspot

from plotting import plot_custom_umap

In [ ]:
adata_epi = adata[adata.obs['coarse_type'] == 'Epithelial'].copy()

In [ ]:
adata_epi.obsm['cellina_basal'] = model.get_latent_representation(adata=adata_epi, latent_key='z')
adata_epi.obsm['cellina_spatial'] = model.get_latent_representation(adata=adata_epi, latent_key='s')
adata_epi.obsm['cellina_joint'] = model.get_latent_representation(adata=adata_epi)

In [ ]:
adata_epi.layers['counts_csc'] = adata_epi.layers['counts']
hs = hotspot.Hotspot(
    adata_epi,
    layer_key="counts_csc",
    model='danb',
    latent_obsm_key="cellina_spatial",
    umi_counts_obs_key="nCount_RNA"
)

hs.create_knn_graph(
    weighted_graph=False, n_neighbors=30,
)

In [ ]:
hs_results = hs.compute_autocorrelations(jobs=10)

In [ ]:
# Select the genes with significant lineage autocorrelation
top_k = 1200
hs_genes = hs_results.loc[hs_results.FDR < 0.05].head(top_k).index

In [ ]:
# Compute pair-wise local correlations between these genes
load_lcz = True
base_dir = os.path.join(DATA_ROOT, "datasets/cosmx/crc_wt_cosmx/analysis")
#lcz_path = f'{base_dir}/210_hotspot_lcz.csv'
lcz_path = f'{base_dir}/210_ligands_hotspot_lcz.csv'

if load_lcz:
    lcz = pd.read_csv(lcz_path, index_col=0)
    hs.local_correlation_z = lcz
else:
    lcz = hs.compute_local_correlations(hs_genes, jobs=10)
    lcz.to_csv(lcz_path)

In [ ]:
modules = hs.create_modules(
    min_gene_threshold=100, core_only=True, fdr_threshold=0.05
)

In [ ]:
module_scores = hs.calculate_module_scores()

module_scores.head()

In [ ]:
module_cols = []
for c in module_scores.columns:
    key = f"Module {c}"
    adata_epi.obs[key] = module_scores[c]
    module_cols.append(key)

In [ ]:
plot_custom_umap(
    adata_epi, subsample=0.2, use_rep='cellina_spatial', color=module_cols,frameon=False, vmin=-1, vmax=1, wspace=0.2
)

In [ ]:
import re

# Define a function to extract the clean status
def extract_status(value):
    # Split by '_' and take the second part
    status = value.split('_')[1]
    # Remove any trailing digits
    status = re.sub(r'\d+$', '', status)
    return status

# Apply to adata.obs
adata_epi.obs['typ_clean'] = adata_epi.obs['typ'].map(extract_status)

In [ ]:
plot_custom_umap(
    adata_epi, subsample=0.2, use_rep='cellina_spatial', color=['typ_clean', 'ist', 'jst'], frameon=False, vmin=-1, vmax=1, wspace=0.2
)

In [ ]:
module_scores_epi = module_scores.loc[adata_epi.obs_names]

In [ ]:
adata_epi.obsm['module_scores'] = module_scores_epi.values

In [ ]:
top_modules = module_scores.idxmax(axis=1)

# Add to adata.obs
adata_epi.obs["top_module"] = top_modules.astype(str)
adata_epi.obs["top_module"] = adata_epi.obs["top_module"].astype("category")

In [ ]:
# Plot
sc.pl.spatial(
    adata_epi,
    color=["top_module", "ist", "typ"],
    palette=None,
    spot_size=50,
    show=True
)

### LR enrichment in modules

In [ ]:
import requests
import io

# read csv from link
# https://github.com/ventolab/cellphonedb-data/blob/master/data/interaction_input.csv
resource = requests.get('https://raw.githubusercontent.com/ventolab/cellphonedb-data/master/data/interaction_input.csv').content
resource = io.StringIO(resource.decode('utf-8'))
resource = pd.read_csv(resource, sep=',')
# keep only PPIs
resource = resource[resource['is_ppi']][['interactors']]
# replace + with _
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('+', '_'))
# if interactors contains two '-' replace the first one with '&
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('-', '&', 1) if x.count('-') == 2 else x)
# split by - and expand
resource = resource['interactors'].str.split('-', expand=True)
# replace & with - in the first column
resource[0] = resource[0].apply(lambda x: x.replace('&', '-'))
resource.columns = ['ligand', 'receptor']

In [ ]:
# Split ligands on '_' to get list of subunits
resource['ligand'] = resource['ligand'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('ligand').reset_index(drop=True)
ligands = resource['ligand'].unique()

In [ ]:
# collect ligands and receptors
ligands = set(resource.ligand)
receptors = set(resource.receptor)

gene_roles = {}
for g in ligands.union(receptors):
    if g in ligands and g in receptors:
        gene_roles[g] = "ligand+receptor"
    elif g in ligands:
        gene_roles[g] = "ligand"
    elif g in receptors:
        gene_roles[g] = "receptor"

In [ ]:
# classify
sig_genes = hs.results.query("Z > 20").index.tolist()
gene_classes = pd.Series(sig_genes, name="gene").map(gene_roles).fillna("none")

# count
counts = gene_classes.value_counts()
print(counts)

In [ ]:
def simplify_class(x):
    if pd.isna(x) or x == "none":
        return ["rest"]
    elif x == "ligand+receptor":
        return ["ligand", "receptor"]
    else:
        return [x]

# Assume gene_roles is a dictionary mapping gene names to roles
hs.results['class'] = hs.results.index.map(gene_roles).map(simplify_class)

In [ ]:
k = 2000

# Sort by 'C' descending and take top k
df_top = hs.results.sort_values("C", ascending=False).head(k).copy()

In [ ]:
collectri = dc.op.collectri(organism="human")

In [ ]:
df_tf = df_top.copy()
tf_genes = set(collectri['source'])

# Ensure list-type entries are handled properly
mask = df_tf.index.isin(tf_genes) & df_tf['class'].apply(lambda x: x == ['rest'])
df_tf.loc[mask, 'class'] = [['TF']]  # keep same list format

In [ ]:
from plotting import plot_autocorr

plot_autocorr(df_top, plot="violin")

## Pathway enrichment

In [ ]:
from plotting import plot_pathway_activity

In [ ]:
# Get module x gene matrix
gene_modules = hs.modules
gene_c = hs.results[['C']]
df = gene_c.join(gene_modules.rename("Module"))
module_gene_matrix = df.pivot_table(index="Module", columns=df.index, values="C", fill_value=0)

In [ ]:
pw_progeny = dc.op.progeny(organism="human")
pw_hallmark = dc.op.hallmark(organism="human")

In [ ]:
# Progeny pathway activity
pw_acts, pw_padj = dc.mt.ulm(data=module_gene_matrix, net=pw_progeny)
plot_pathway_activity(pw_acts, pw_padj, alpha=0.05)

In [ ]:
# Hallmark pathway activity
pw_acts, pw_padj = dc.mt.ulm(data=module_gene_matrix, net=pw_hallmark)
plot_pathway_activity(pw_acts, pw_padj, alpha=0.05)

### Pathway targets

In [ ]:
sc.pp.log1p(adata_epi)
sc.tl.rank_genes_groups(adata_epi, groupby="top_module")
deg = sc.get.rank_genes_groups_df(adata_epi, group="1").set_index("names")

In [ ]:
dc.pl.source_targets(data=deg, x="weight", y="scores", net=pw_progeny, name="NFkB", top=15, figsize=(5, 6))

In [ ]:
dc.pl.source_targets(data=deg, x="weight", y="scores", net=pw_progeny, name="TNFa", top=15, figsize=(5, 6))

In [ ]:
dc.pl.source_targets(data=deg, x="weight", y="scores", net=pw_progeny, name="Hypoxia", top=15, figsize=(5, 6))

## Deeplift

In [ ]:
from interpretability import run_deeplift, add_deeplift_to_obs

In [ ]:
# compute attributions for the following genes
gene_indices = list([0, 3, 5])
baseline = "mean"

adata_dummy = adata[:100].copy()
attrs = run_deeplift(
    model=model,
    adata=adata_dummy,
    target_genes=gene_indices,
    batch_size=512,
    baseline=baseline,
)

print(attrs.shape) # -> (n_cells, n_spatial_features, n_target_genes)

In [ ]:
adata_dummy.obsm["deeplift_spatial"] = attrs.numpy()

In [ ]:
adata.uns["deeplift_spatial"] = {
    "axes": ["cells", "spatial_features", "genes"],
    "gene_indices": gene_indices,
    "gene_names": adata_dummy.var_names[gene_indices].tolist(),
    "spatial_feature_names": list(adata_dummy.uns["spatial_x_names"]),
    "method": "DeepLift",
    "target": "px_rate",
    "baseline": f"{baseline}",
}

In [ ]:
add_deeplift_to_obs(adata_dummy, "ABCB1", agg="l2")